In [0]:
order_number= spark.read.table("retail_data_platform.silver.sales_orders")
sales = spark.read.table("retail_data_platform.silver.sales")
import pyspark.sql.functions as f
#retail_data_platform.silver.promotions
#retail_data_platform.silver.orders_products
#retail_data_platform.silver.sales

In [0]:
%sql
select customer_id,price , Rank() over(partition by customer_id order by price desc) as price_rank
from retail_data_platform.silver.orders_products


In [0]:
%sql
with ranked as (
    select customer_id, unit,price, qty ,row_number() over(partition by customer_id order by price desc) as rn
    from retail_data_platform.silver.orders_products
)
select * from ranked where rn <=2
-- finding the top 2 orders per customers

In [0]:
%sql
select * from retail_data_platform.silver.orders_products

In [0]:
%sql
select product_name , count(*) from retail_data_platform.silver.orders_products
group by product_name
order by count(*) desc

In [0]:
%sql 
WITH ranked AS (
    SELECT 
        customer_id, 
        product_name, 
        SUM(price * qty) AS spending 
    FROM retail_data_platform.silver.orders_products
    GROUP BY 
        customer_id, 
        product_name
    HAVING 
        product_name = '15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)'
        AND SUM(price * qty) >= 1000
)
SELECT 
    customer_id, 
    product_name, 
    spending, 
    rank() OVER (PARTITION BY customer_id ORDER BY spending DESC) AS rn
FROM ranked;

In [0]:
%sql
with ranked as (
    select customer_id , product_name , sum(price*qty) as spending 
    from retail_data_platform.silver.orders_products
    group by customer_id, product_name
    having product_name == "15.4 NakBook Pro with Touch Bar (Late 2016, Space Gray)"
    and Sum(price*qty) >= 1000
)

select customer_id , product_name , spending , 
row_number() over(partition by customer_id order by spending desc) as rn
from ranked

In [0]:
from pyspark.sql import Window

df = spark.read.table("retail_data_platform.silver.orders_products")
window = (Window.partitionBy(f.col("customer_id")).orderBy(f.col("price").desc()))
results = df.withColumn("rn",f.row_number().over(window)).alias("top_2").filter(f.col("rn")<=2)
results.select("customer_id","price","rn").display()

In [0]:
from pyspark.sql import Window

df = spark.read.table("retail_data_platform.silver.orders_products")


window = (Window.partitionBy(f.col("customer_id")).orderBy(f.col("price").desc()))
dff = df.select("customer_id","price",f.rank().over(window).alias("Rank by Price"))
display(dff)

In [0]:
%sql

select customer_id , count("order_number") from retail_data_platform.silver.sales_orders
group by customer_id




In [0]:
%sql
select * from retail_data_platform.silver.sales_orders
where customer_id == 22100013 and order_number == 317569238 ;


In [0]:
fil = order_number.filter((f.col("customer_id")==22100013) & (f.col("order_number")==317569238))
                                  
fil.show()

In [0]:

get_products = order_number.groupBy(f.col("customer_id")).agg(f.count("*")).alias("rows_count").show()

In [0]:
%sql
select customer_id , order_number , count(*) as total_rowss from retail_data_platform.silver.sales_orders

group by customer_id ,order_number
having total_rowss > 1

In [0]:
dff = order_number.groupBy(f.col("customer_id"),f.col("order_number")).agg(f.count("*")).filter(f.count("*")>1).show()

In [0]:
%sql
create schema if not exists retail_data_platform.gold

In [0]:
%sql 
create or replace table retail_data_platform.gold.dim_customer (
    customer_id bigint not null,
    first_name string,
    last_name string,
    full_name string,
    loyalty_segment int,
    created_date timestamp,
    last_update_ts timestamp,
    constraint pk_dim_customers
        primary key(customer_id)
);
 
insert into retail_data_platform.gold.dim_customer (
    customer_id,
    first_name,
    last_name,
    full_name,
    loyalty_segment,
    created_date,
    last_update_ts

)
select
    customer_id ,
    first_name ,
    last_name,
    customer_name as full_name,
    loyalty_segment,
    valid_from as created_date,
    last_update_ts
from retail_data_platform.silver.customers

 



